# ipywidget

In [1]:
# !uv pip list|egrep "huggingface|hf-xet" 

!jupyter nbextension enable --py --sys-prefix widgetsnbextension

Enabling notebook extension jupyter-js-widgets/extension...
      - Validating: OK


In [2]:
import ipywidgets as widgets
from IPython.display import display
from ipywidgets import interact

#default_model_id = 'unsloth/Qwen-AgentWorld-35B-A3B-GGUF'  # kaggle 2xT4 59tps
default_model_id = 'unsloth/Qwen3.8-27B-GGUF:Q4_K_M'

model_id_list = [
     default_model_id,
    'deepreinforce-ai/Ornith-1.0-9B-GGUF',  # kaggle CPU 3tps
    'deepreinforce-ai/Ornith-1.0-35B-GGUF',
    'Jackrong/Qwopus3.6-27B-v2-GGUF:Q3_K_M',
    'InternScience/Agents-A1-Q4_K_M-GGUF',
    'unsloth/Qwen3.8-27B-GGUF:Q5_K_M',
    'unsloth/Qwen3.8-27B-GGUF:UD-Q5_K_M',
    'unsloth/Qwen3.8-27B-GGUF:Q4_K_M',
    'unsloth/Qwen3.8-27B-GGUF:UD-Q4_K_M',
]

# assert default_model_id in model_id_list, "not in model_id_list"
if default_model_id not in model_id_list:
    print(
        f"⚠️ default_model_id {default_model_id} not in model_id_list {model_id_list},"
        " set to model_id_list[0]"
    )
    default_model_id = model_id_list[0]

def ineract_f(model_id):
    print(f"💡 {model_id=}")

# Create the dropdown
model_widget = widgets.Dropdown(
    options=model_id_list,
    value=default_model_id,
    description='Pick a model:',
    disabled=False,
)

# Display the dropdown
# display(dropdown)

_ = interact(ineract_f, model_id=model_widget)

interactive(children=(Dropdown(description='Pick a model:', options=('unsloth/Qwen3.8-27B-GGUF:Q4_K_M', 'deepr…

In [3]:
model_id = model_widget.value
# print(f'{model_id=}')

# prelim
HF_TOKEN
HF_XET_HIGH_PERFORMANCE

In [4]:
import os
import subprocess as sp
from shlex import split
from pathlib import Path

if not os.getenv("HF_TOKEN"):
  try:
    from set_env import set_env
  except ModuleNotFoundError:
    !uv pip install set-env-colab-kaggle-dotenv
    from set_env import set_env
  set_env("HF_TOKEN")
    
if not os.getenv("HF_TOKEN"):
  print(
      "⚠️HF_TOKEN not set in Add-ons/Secrets or otherwise. "
      "But you can safely ignore the previous error messages. "
      "You will just spend a bit longer to download the model files from huggingface.\n"
  )
    
!uv pip install huggingface_hub hf-xet

%env HF_XET_HIGH_PERFORANCE=1

has_gpu = not not sp.run(split("ls /dev/nvidia0"), capture_output=1, check=0).stdout

f'💡 {has_gpu=}'


Using Python 3.12.13 environment at: /usr
Resolved 3 packages in 267ms                                         
Prepared 2 packages in 31ms                                              
Installed 2 packages in 7ms-dotenv==0.1.4                   
 + loguru==0.7.3
 + set-env-colab-kaggle-dotenv==0.1.4
Using Python 3.12.13 environment at: /usr
Checked 2 packages in 135ms
env: HF_XET_HIGH_PERFORANCE=1


'💡 has_gpu=True'

deepreinforce-ai/Ornith-1.0-35B-GGUF

Jackrong/Qwopus3.5-27B-v3

https://github.com/ggml-org/llama-install.sh/blob/master/REQUIREMENTS.md

linux	x86_64	cuda	GLIBC 2.35	libcuda.so.1

!ldd --version

!sudo find / -name "libcuda.so.1" 2>/dev/null

!ln -sf /usr/local/nvidia/lib64/libcuda.so /usr/local/cuda/lib64/libcuda.so

# install llama-app or pre-built llama-server

In [5]:
%%time
# Version: b9821 b9835
# colab Wall time: 5.75 s

if has_gpu:
    # install precompiled llama-server
    import subprocess as sp
    
    if not sp.run("command -v kaggle", shell=1, capture_output=1).stdout:
      !uv pip install kaggle
    
    if not sp.run("command -v llama-server", shell=1, capture_output=1).stdout:
      !kaggle datasets download -d mikeee8/llama-cpp-precompiled-202606 --unzip
      !ls -l
      !install -t /usr/local/bin llama-*

    env = os.environ.copy()
    !command -v llama-server    

else:
    !curl -LsSf https://llama.app/install.sh | SKIP_ROMC=1 SKIP_VULKAN=1 sh
    
    !ls ~/.llama-app/llama -l
    !ls ~/.local/bin -l
    
    # !echo $HOME/.local/bin
    from pathlib import Path
    import os
    local_bin = Path("~/.local/bin").expanduser().as_posix()
    
    if local_bin not in os.getenv("PATH"):
        os.environ['PATH'] = f"{local_bin}:{os.getenv('PATH')}"
    
    display(os.getenv("PATH"))
    
    env = os.environ.copy()
    !command -v ollama

Dataset URL: https://www.kaggle.com/datasets/mikeee8/llama-cpp-precompiled-202606
License(s): CC0-1.0
100%|█████████████████████████████████████████| 305M/305M [00:01<00:00, 163MB/s]

total 837772
-rw-r--r-- 1 root root 215713288 Sep 10 01:19 llama-cli
-rw-r--r-- 1 root root 208602200 Sep 10 01:19 llama-gguf-split
-rw-r--r-- 1 root root 214600416 Sep 10 01:19 llama-mtmd-cli
-rw-r--r-- 1 root root 218953920 Sep 10 01:19 llama-server
/usr/local/bin/llama-server
CPU times: user 108 ms, sys: 45 ms, total: 153 ms
Wall time: 10.3 s


In [6]:
_ = '''
import os

# os.environ['PATH'] = '/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin'
if "~/.local/bin" not in os.getenv("PATH"):
  %env PATH=~/.local/bin:{os.getenv("PATH")}

!echo $PATH
!command -v llama
# '''


# run llama serve or llama-server

In [7]:
if has_gpu:
    # run llama-server
    import subprocess as sp
    
    from shlex import split
    # cmd = 'llama-server -hf Jackrong/Qwopus3.6-27B-v2-GGUF:Q4_K_M'
    # cmd = "llama serve -hf Jackrong/Qwopus3.6-27B-v2-GGUF:Q2_K --port 8088 --verbose"  # crash 10GB
    
    # model = 'Jackrong/Qwopus3.6-27B-v2-GGUF:Q2_K'
    
    # model = 'deepreinforce-ai/Ornith-1.0-9B-GGUF'
    # colab T1 13.4GB VRAM 35tps， ctx-size: 262,144
    
    # cmd = f"llama serve -hf {model_id} -c 8196 --port 8088 --verbose -ngl 99"
    #cmd = f"llama-server -hf {model_id} --port 8088 -ngl all -fa on -np 1 -1 -a {model_id.split('/')[-1].lower()}"
    
    os.environ["GGML_CUDA_P2P"] = "1"
    
    cmd = (
        f"llama-server "
        f"-hf {model_id} "
        f"--port 8088 "
        f"-ngl all "
        f"-sm tensor "
        f"-fa on "
        f"-c 32768 "
        f"-np 1 "
        f"--alias qwen3.8-27b"
    )
    
    cmd_v = split(cmd)
    
    _ = """
    with open("llama-serve.txt", 'w') as log:
        proc_llama_serve = sp.Popen(
            cmd_v,
            start_new_session=1,
            stdout=log,
            stderr=log,
        )
    
    # """

else:
    import subprocess as sp
    
    from shlex import split
    # cmd = 'llama serve -hf Jackrong/Qwopus3.6-27B-v2-GGUF:Q4_K_M'
    
    # Qwopus3.6-27B-v2-Q3_K_M.gguf
    # model = 'Jackrong/Qwopus3.6-27B-v2-GGUF:Q3_K_M'
    
    # model = 'deepreinforce-ai/Ornith-1.0-9B-GGUF'  # kaggle CPU 3tps
    # model = 'deepreinforce-ai/Ornith-1.0-35B-GGUF'
    
    cmd = f"llama serve -hf {model_id} --port 8088 -a {model_id.split('/')[-1].lower()}"
    
    cmd_v = split(cmd)

# ############
# _ = """
with open("llama-serve.txt", 'w') as log:
    c = sp.Popen(
        cmd_v,
        start_new_session=1,
        stdout=log,
        stderr=log,
        env=env,
    )
# """

print(f"💡 {cmd=}")

💡 cmd='llama-server -hf unsloth/Qwen3.8-27B-GGUF:Q4_K_M --port 8088 -ngl all -sm tensor -fa on -c 32768 -np 1 --alias qwen3.8-27b'


In [8]:
!ls -lrt

total 837772
-rw-r--r-- 1 root root 215713288 Sep 10 01:19 llama-cli
-rw-r--r-- 1 root root 208602200 Sep 10 01:19 llama-gguf-split
-rw-r--r-- 1 root root 214600416 Sep 10 01:19 llama-mtmd-cli
-rw-r--r-- 1 root root 218953920 Sep 10 01:19 llama-server
-rw-r--r-- 1 root root         0 Sep 10 01:19 llama-serve.txt


In [9]:
!pgrep llama
# !pkill llama
# proc_llama_serve.terminate()

!ps aux|grep llama|grep -Ev "grep|defunct"

138
root         138  0.0  0.2 1632256 69216 ?       Ds   01:19   0:00 llama-server -hf unsloth/Qwen3.8-27B-GGUF:Q4_K_M --port 8088 -ngl all -sm tensor -fa on -c 32768 -np 1 --alias qwen3.8-27b


In [10]:
print(f'{model_id=}')

# !ls -l /root/.cache/huggingface/hub
# !du -sh /root/.cache/huggingface/hub/*

model_id='unsloth/Qwen3.8-27B-GGUF:Q4_K_M'


#### cmd_v

In [11]:
!tail llama-serve.txt

!grep "server is listening" llama-serve.txt 


# llama_server: server is listening on http://127.0.0.1:8080

In [12]:
!curl -sS 127.0.0.1:8088/health

curl: (7) Failed to connect to 127.0.0.1 port 8088 after 0 ms: Connection refused


In [13]:
print(f"{cmd=}")
# !{cmd}


cmd='llama-server -hf unsloth/Qwen3.8-27B-GGUF:Q4_K_M --port 8088 -ngl all -sm tensor -fa on -c 32768 -np 1 --alias qwen3.8-27b'


# cloudflare_tunnel

In [14]:
# import os
import re
import shutil
import subprocess as sp
import time
from pathlib import Path


def cloudflare_tunnel(port: int = 8001) -> str:
    """
    Ensure cloudflared is installed, checks for active instances running.

    on the target port using ps aux pipelines, and provisions a secure tunnel.
    """
    tunnel_log = Path("tunnel-log.txt")
    tunnel_log.touch()

    # --- 1. Check for Running Processes using ps aux Pipeline ---
    # Construct the query matching your requested format
    ps_cmd = "ps aux | grep cloudflared | grep -v grep"
    ps_output = ""

    try:
        ps_output = sp.check_output(ps_cmd, shell=True).decode()
    except sp.CalledProcessError:
        pass  # No running cloudflared instances found

    if ps_output:
        # Split output line by line to evaluate individual process instances
        lines = [line.strip() for line in ps_output.split("\n") if line.strip()]

        for line in lines:
            # Extract the Process ID (PID is the second column in ps aux)
            parts = line.split()
            if len(parts) < 2:
                continue
            pid = parts[1]

            # Look for the port sequence within this process execution string
            port_match = re.search(r"--url http://localhost:(\d+)", line)
            running_port = port_match.group(1) if port_match else "unknown"

            # Check if this specific instance matches our target configuration port
            if running_port == str(port):
                print(
                    f"⚠️ Found cloudflared running on port {port} (PID: {pid})."
                )
                print(f"Command context: {line}")


                print(f" process {pid} eixists that seem to forward to the same address")
                # sp.run(f"kill -9 {pid}", shell=True)

                print(
                    "🔄 Attempting to retrieve URL from the existing log file..."
                )
                if tunnel_log.exists():
                    log_text = tunnel_log.read_text()
                    urls = re.findall(
                        r"https://[a-z-]+\.trycloudflare\.com", log_text
                    )
                    if urls:
                        print(f"the url is {urls[-1]}")
                        return urls[-1]
                print(
                    "❌ Could not locate active domain in current log.  exiting..."
                )
                return ""

    # --- 2. Download and Install cloudflared if Missing ---
    if not shutil.which("cloudflared"):
        print("cloudflared not found. Downloading and installing...")
        download_cmd = "wget -q -nv -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared"
        sp.run(download_cmd, shell=True, check=True)

        _ = """
        # kaggle detect
        if 'kaggle' in str(os.environ).lower():
            # Correctly resolve the absolute path for the directory
            local_bin_dir = Path("~/.local/bin").expanduser()
            local_bin_dir.mkdir(parents=True, exist_ok=True)

            # Use explicit target path in install string
            install_cmd = f"install cloudflared {local_bin_dir}/cloudflared"
            sp.run(install_cmd, shell=True, check=True)

            # CRITICAL: inject the directory into Python's path so section 3 works
            if str(local_bin_dir) not in os.environ["PATH"]:
                os.environ["PATH"] += os.pathsep + str(local_bin_dir)
        else:
        # """

        install_cmd = "install -t /usr/local/bin cloudflared"
        sp.run(install_cmd, shell=True, check=True)
        print("✅ cloudflared installed successfully.")
    else:
        print("💡 cloudflared is already installed. Skipping download.")

    # --- 3. Spin up New Cloudflare Tunnel ---
    if tunnel_log.exists():
        tunnel_log.unlink()

    _ = """
    tunnel_cmd = f"cloudflared tunnel --url http://localhost:{port} > tunnel-log.txt 2>&1"
    sp.Popen(tunnel_cmd, shell=True)
    # """

    # _ = '''
    # Open the log file safely
    with open("tunnel-log.txt", "a") as log_file:
        # Python runs this in the background automatically, no shell needed
        proc = sp.Popen(
            ["cloudflared", "tunnel", "--url", f"http://localhost:{port}"],
            stdout=log_file,
            stderr=log_file
        )
        cloudflare_tunnel.proc = proc  # type: ignore

    # '''

    # --- 4. Dynamic URL Capture Polling ---
    for _ in range(10):
        if tunnel_log.exists():
            log_text = tunnel_log.read_text()
            urls = re.findall(r"https://[a-z-]+\.trycloudflare\.com", log_text)
            if urls:
                url_ = urls[-1]
                print("✍️ tunnel-log.txt")
                print(f"💡 \n\t{url_}")
                print(f"👉fwd to http://localhost:{port}")
                return url_
        time.sleep(1)

    # Failure debugging block
    if tunnel_log.exists():
        print("\n--- Tunnel Error Logs ---")
        print(tunnel_log.read_text())


    return ".trycloudflare.com not found, something went wrong"


In [15]:
url8088 = cloudflare_tunnel(8088)

cloudflared not found. Downloading and installing...
✅ cloudflared installed successfully.
✍️ tunnel-log.txt
💡 
	https://admitted-playstation-dana-masters.trycloudflare.com
👉fwd to http://localhost:8088


In [16]:
!pgrep cloudflared
# !pkill cloudflared
!ps aux|grep cloudflared  | grep -Ev "grep|defunct"

161
root         161  6.6  0.1 1294740 39384 ?       Sl   01:19   0:00 cloudflared tunnel --url http://localhost:8088


In [17]:
import subprocess as sp

# default port 8000
sp.run("nohup python -m http.server > http_server-log.txt 2>&1 &", shell=1, check=1)

CompletedProcess(args='nohup python -m http.server > http_server-log.txt 2>&1 &', returncode=0)

In [18]:
!curl 127.0.0.1:8000 -sSI
!tail http_server-log.txt
# !ls -l

curl: (7) Failed to connect to 127.0.0.1 port 8000 after 0 ms: Connection refused


# wait for server to come up

In [19]:
from tqdm import tqdm
from types import SimpleNamespace

ns = SimpleNamespace(flag=False)

def wait_for(timeout=600, cb=None, msg='flagged'):
    if cb is None:
        cb = lambda: None
    try:
        for _ in tqdm(range(timeout)):
            # if ns.flag:
            if cb():
                print(f"✅ {msg}")
                break
            time.sleep(1)
        else:
            print(f"❌timed out {timeout} s")
    except KeyboardInterrupt:
        print('💡 interrupted')

In [20]:
print("💡 downloading model files, can take a while dependent on file size")

cb = lambda: sp.run(split("curl 127.0.0.1:8088/v1/models -sS"), capture_output=1).stdout
wait_for(cb=cb, msg='local server up running')

💡 downloading model files, can take a while dependent on file size


 30%|███       | 181/600 [03:02<07:03,  1.01s/it]

✅ local server up running


In [21]:
cb = lambda: sp.run(split(f"curl {url8088} -sS"), capture_output=1).stdout
wait_for(cb=cb, msg=f'tunnel {url8088} ready')

print(f"\n💡 browse to 👉 {url8088} to chat (if you see garbled text, it's because the model is not fully loaded, wait for a while and refresh)")

print(f"\n💡 model list: {url8088}/v1/models")
print(f"\n💡 base_url: {url8088}/v1")

  0%|          | 0/600 [00:00<?, ?it/s]

✅ tunnel https://admitted-playstation-dana-masters.trycloudflare.com ready

💡 browse to 👉 https://admitted-playstation-dana-masters.trycloudflare.com to chat (if you see garbled text, it's because the model is not fully loaded, wait for a while and refresh)

💡 model list: https://admitted-playstation-dana-masters.trycloudflare.com/v1/models

💡 base_url: https://admitted-playstation-dana-masters.trycloudflare.com/v1


# check Interactive env

In [22]:
# check Interactive env

import os

# !env | grep -i interact

if 'Interactive' in str(os.environ):
    print("💡 interactive env")
else:
    print("💡 non-interactive env")
    # !sleep infinity
    import signal
    import time
    from types import SimpleNamespace
    
    state = SimpleNamespace(running=True)
    
    def signal_handler(sig, frame):
        print('💡 Shutting down gracefully...')
        state.running = False
    
    signal.signal(signal.SIGINT, signal_handler)
    signal.signal(signal.SIGTERM, signal_handler)
    
    while state.running:
        time.sleep(5)

💡 interactive env
